# Project 3: Surgical Outcomes & Cardiovascular Risk
## NLP on Clinical Notes + Structured Data Fusion

---

**Objective:** Build an NLP pipeline to extract clinical comorbidities from surgical notes, then fuse those extracted features with structured heart disease data to predict cardiovascular risk in surgical patients.

**Core Question:** Does parsing clinical notes add predictive value beyond structured data alone?

**Datasets:**
- Anesthesia Dataset (300 patients) — surgical records with free-text clinical notes
- Heart Disease UCI (920 patients) — structured cardiovascular risk factors

---

### Executive Summary
*(Fill this in after completing the project)*

- **Problem:**
- **Approach:**
- **NLP Extraction Accuracy (F1):**
- **NLP Uplift (AUC improvement):**
- **Key Finding:**
- **Limitations:**

In [ ]:
# Prompt for Claude Code:
#
# Generate the setup and Step 1 exploration code for this notebook.
#
# 1. **Setup cell**: Import pandas, numpy, re, random, matplotlib, seaborn,
#    scipy.stats (chi2_contingency, ttest_ind), and sklearn (train_test_split,
#    LogisticRegression, RandomForestClassifier, classification_report,
#    confusion_matrix, roc_auc_score, roc_curve, accuracy_score, SimpleImputer,
#    LabelEncoder). Set plot defaults (10x6 figure, dpi 100, whitegrid style).
#    Set random seeds to 42 for random, numpy.
#
# 2. **Load both datasets**: Read Anesthesia_Dataset.csv into `anes` and
#    heart_disease_uci.csv into `heart`. Print shape, column count, and
#    memory usage for each.
#
# 3. **Anesthesia overview**: Print dtypes, missing value counts (only columns
#    with missing values), and display head().
#
# 4. **Heart Disease overview**: Print dtypes, missing value counts sorted
#    ascending (only columns with missing), overall missing percentage, and
#    display head().

In [12]:
#1
import pandas as pd
import numpy as np
import re
import random
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, ttest_ind
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report,
  confusion_matrix,
                               roc_auc_score, roc_curve,
accuracy_score)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

# Set plot defaults
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

# Set random seeds
random.seed(42)
np.random.seed(42)

In [13]:
#2
# Load datasets
anes = pd.read_csv('Anesthesia_Dataset.csv')
heart = pd.read_csv('heart_disease_uci.csv')

# Print information for Anesthesia Dataset
print("="*50)
print("ANESTHESIA DATASET")
print("="*50)
print(f"Shape: {anes.shape}")
print(f"Column count: {len(anes.columns)}")
print(f"Memory usage:\n{anes.memory_usage(deep=True)}\n")

# Print information for Heart Disease Dataset
print("="*50)
print("HEART DISEASE DATASET")
print("="*50)
print(f"Shape: {heart.shape}")
print(f"Column count: {len(heart.columns)}")
print(f"Memory usage:\n{heart.memory_usage(deep=True)}")

ANESTHESIA DATASET
Shape: (300, 12)
Column count: 12
Memory usage:
Index                   132
PatientID              2400
Age                    2400
Gender                15000
BMI                    2400
SurgeryType           17976
SurgeryDuration       16729
AnesthesiaType        16498
PreoperativeNotes     21014
PostoperativeNotes    22072
PainLevel              2400
Complications         17823
Outcome                2400
dtype: int64

HEART DISEASE DATASET
Shape: (920, 16)
Column count: 16
Memory usage:
Index         132
id           7360
age          7360
sex         49148
dataset     53820
cp          56530
trestbps     7360
chol         7360
fbs         32760
restecg     53848
thalch       7360
exang       32900
oldpeak      7360
slope       43727
ca           7360
thal        41810
num          7360
dtype: int64


In [14]:
# 3
# Anesthesia Dataset Overview
print("="*50)
print("ANESTHESIA DATASET OVERVIEW")
print("="*50)

# Print dtypes
print("\nData Types:")
print(anes.dtypes)

# Print missing values (only columns with missing values)
print("\nMissing Values (columns with missing data):")
missing_values = anes.isnull().sum()
missing_values = missing_values[missing_values > 0]
if len(missing_values) > 0:
  print(missing_values)
else:
  print("No missing values found")

# Display head
print("\nFirst 5 rows:")
print(anes.head())

ANESTHESIA DATASET OVERVIEW

Data Types:
PatientID             int64
Age                   int64
Gender                  str
BMI                   int64
SurgeryType             str
SurgeryDuration         str
AnesthesiaType          str
PreoperativeNotes       str
PostoperativeNotes      str
PainLevel             int64
Complications           str
Outcome               int64
dtype: object

Missing Values (columns with missing data):
Complications    73
dtype: int64

First 5 rows:
   PatientID  Age Gender  BMI     SurgeryType SurgeryDuration AnesthesiaType  \
0          1   33      M   32    Neurological         217 min          Local   
1          2   33      M   23  Cardiovascular         181 min          Local   
2          3   58      F   24      Orthopedic          79 min        General   
3          4   65      F   26      Orthopedic         210 min          Local   
4          5   65      M   28    Neurological         221 min        General   

        PreoperativeNotes          

In [19]:
#4
# Heart Disease Dataset Overview
print("="*50)
print("HEART DISEASE DATASET OVERVIEW")
print("="*50)

# Print dtypes
print("\nData Types:")
print(heart.dtypes)

# Print missing values (only columns with missing values)
print("\nMissing Values (columns with missing data):")
missing_values = heart.isnull().sum()
missing_values = missing_values[missing_values > 0]
if len(missing_values) > 0:
  print(missing_values)
else:
  print("No missing values found")

# Display head
print("\nFirst 5 rows:")
print(heart.head())

HEART DISEASE DATASET OVERVIEW

Data Types:
id            int64
age           int64
sex             str
dataset         str
cp              str
trestbps    float64
chol        float64
fbs          object
restecg         str
thalch      float64
exang        object
oldpeak     float64
slope           str
ca          float64
thal            str
num           int64
dtype: object

Missing Values (columns with missing data):
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
dtype: int64

First 5 rows:
   id  age     sex    dataset               cp  trestbps   chol    fbs  \
0   1   63    Male  Cleveland   typical angina     145.0  233.0   True   
1   2   67    Male  Cleveland     asymptomatic     160.0  286.0  False   
2   3   67    Male  Cleveland     asymptomatic     120.0  229.0  False   
3   4   37    Male  Cleveland      non-anginal     130.0  250.0  False   
4   5   41  Female 

### 🔍 Critical Discovery — Examine the Text Fields

Run the cells below and document what you find. How many unique values does each text column have?

In [ ]:
# Prompt for Claude Code:
#
# Generate 2 code cells for the Critical Discovery section:
#
# 1. **Examine text columns**: For PreoperativeNotes and PostoperativeNotes,
#    show value_counts() and nunique(). For Complications, show value_counts()
#    plus missing count and missing percentage.
#
# 2. **Feature overlap**: Print the age ranges for both datasets
#    (anes['Age'] vs heart['age']). Then build a display DataFrame mapping
#    overlapping features between datasets:
#      - Age ↔ age (direct match)
#      - BMI ↔ — (no direct match in heart)
#      - PreoperativeNotes "Hypertension" ↔ trestbps (blood pressure)
#      - PreoperativeNotes "diabetes" ↔ fbs (fasting blood sugar)
#      - SurgeryType=Cardiovascular ↔ num (disease severity)

In [26]:
print("="*50)
print("EXAMINING TEXT COLUMNS")
print("="*50)

# PreoperativeNotes
print("\nPreoperativeNotes:")
preop_unique = anes['PreoperativeNotes'].nunique()
print(f"Unique values (nunique): {preop_unique}")
print("Value counts:")
print(anes['PreoperativeNotes'].value_counts())

# PostoperativeNotes
print("\nPostoperativeNotes:")
postop_unique = anes['PostoperativeNotes'].nunique()
print(f"Unique values (nunique): {postop_unique}")
print("Value counts:")
print(anes['PostoperativeNotes'].value_counts())

# Complications
print("\nComplications:")
missing_count = anes['Complications'].isnull().sum()
missing_pct = (missing_count / len(anes)) * 100
print(f"Missing count: {missing_count}")
print(f"Missing percentage: {missing_pct:.2f}%")
print("Value counts:")
print(anes['Complications'].value_counts())

EXAMINING TEXT COLUMNS

PreoperativeNotes:
Unique values (nunique): 2
Value counts:
PreoperativeNotes
Hypertension, diabetes    157
Stable, no allergies      143
Name: count, dtype: int64

PostoperativeNotes:
Unique values (nunique): 2
Value counts:
PostoperativeNotes
Minimal pain, no complications    152
Pain, slow recovery               148
Name: count, dtype: int64

Complications:
Missing count: 73
Missing percentage: 24.33%
Value counts:
Complications
Respiratory distress     83
Nausea, mild bleeding    80
Delayed recovery         64
Name: count, dtype: int64


### ✏️ Your Observations

**What did you discover about the text columns?**

*Write your observations here:*
- PreoperativeNotes has only ___ unique values: ...
- PostoperativeNotes has only ___ unique values: ...
- This means that for NLP to be meaningful, we'll need to...


---
## Step 2 — Clean the Structured Data

Before we touch the text, let's get the structured data into shape.

### Anesthesia Dataset Cleaning

In [ ]:
# Prompt for Claude Code:
#
# Clean the anesthesia dataset (3 code cells):
#
# 2a. Parse SurgeryDuration (string like '217 min') into hours.
#     Create `duration_hours` = extracted integer / 60.
#     Print before/after examples and the range in hours.
#
# 2b. Handle 73 missing Complications values:
#     Create `has_complications` = 1 if Complications is not null, 0 otherwise.
#     Create `complication_type` = original value filled with 'None' for NaN.
#     Print the distribution of has_complications.
#
# 2c. Encode categoricals:
#     - `gender_encoded`: 1 if Gender == 'M', else 0
#     - `anesthesia_general`: 1 if AnesthesiaType == 'General', else 0
#     - One-hot encode SurgeryType using pd.get_dummies with prefix='surgery'
#       (this creates columns: surgery_Cardiovascular, surgery_Cosmetic,
#        surgery_Neurological, surgery_Orthopedic)
#     Concatenate the dummies onto the anes DataFrame. Print the new columns.

In [31]:
#2a
# Parse SurgeryDuration into hours
print("="*50)
print("PARSING SURGERY DURATION")
print("="*50)

# Show before examples
print("\nBefore (first 5 values):")
print(anes['SurgeryDuration'].head())

# Extract the numeric part and convert to hours
anes['duration_hours'] = anes['SurgeryDuration'].str.extract('(\d+)')[0].astype(int) / 60

# Show after examples
print("\nAfter (first 5 values):")
print(anes['duration_hours'].head())

# Show the range in hours
print(f"\nDuration range (in hours):{anes['duration_hours'].min():.2f} to {anes['duration_hours'].max():.2f}")

PARSING SURGERY DURATION

Before (first 5 values):
0    217 min
1    181 min
2     79 min
3    210 min
4    221 min
Name: SurgeryDuration, dtype: str

After (first 5 values):
0    3.616667
1    3.016667
2    1.316667
3    3.500000
4    3.683333
Name: duration_hours, dtype: float64

Duration range (in hours):1.00 to 4.00


<>:11: SyntaxWarning: invalid escape sequence '\d'
<>:11: SyntaxWarning: invalid escape sequence '\d'
C:\Users\niyat\AppData\Local\Temp\ipykernel_33400\77132635.py:11: SyntaxWarning: invalid escape sequence '\d'
  anes['duration_hours'] = anes['SurgeryDuration'].str.extract('(\d+)')[0].astype(int) / 60


In [36]:
#2a
# Handle missing Complications values
print("="*50)
print("HANDLING MISSING COMPLICATIONS")
print("="*50)

# Create has_complications binary variable (1 if not null, 0 if null)
anes['has_complications'] = anes['Complications'].notna().astype(int)

# Create complication_type with 'None' for missing values
anes['complication_type'] = anes['Complications'].fillna('None')

# Print distribution of has_complications
print("\nDistribution of has_complications:")
print(anes['has_complications'].value_counts())

print("\nPercentages:")
print(anes['has_complications'].value_counts(normalize=True)
* 100)

HANDLING MISSING COMPLICATIONS

Distribution of has_complications:
has_complications
1    227
0     73
Name: count, dtype: int64

Percentages:
has_complications
1    75.666667
0    24.333333
Name: proportion, dtype: float64


In [38]:
#2c
# Encode categorical variables
print("="*50)
print("ENCODING CATEGORICAL VARIABLES")
print("="*50)

# Create gender_encoded: 1 if 'M', 0 otherwise
anes['gender_encoded'] = (anes['Gender'] == 'M').astype(int)

# Create anesthesia_general: 1 if 'General', 0 otherwise
anes['anesthesia_general'] = (anes['AnesthesiaType'] == 'General').astype(int)

# One-hot encode SurgeryType
surgery_dummies = pd.get_dummies(anes['SurgeryType'], prefix='surgery')

# Concatenate dummies onto the DataFrame
anes = pd.concat([anes, surgery_dummies], axis=1)

# Print the new columns
print("\nNew columns added:")
new_cols = ['gender_encoded', 'anesthesia_general'] + list(surgery_dummies.columns)
print(new_cols)

# Show the new encoded columns
print("\nFirst 5 rows of new encoded columns:")
print(anes[new_cols].head())

ENCODING CATEGORICAL VARIABLES

New columns added:
['gender_encoded', 'anesthesia_general', 'surgery_Cardiovascular', 'surgery_Cosmetic', 'surgery_Neurological', 'surgery_Orthopedic']

First 5 rows of new encoded columns:
   gender_encoded  anesthesia_general  surgery_Cardiovascular  \
0               1                   0                   False   
1               1                   0                    True   
2               0                   1                   False   
3               0                   0                   False   
4               1                   1                   False   

   surgery_Cosmetic  surgery_Neurological  surgery_Orthopedic  
0             False                  True               False  
1             False                 False               False  
2             False                 False                True  
3             False                 False                True  
4             False                  True               False  


### Heart Dataset Cleaning

In [ ]:
# Prompt for Claude Code:
#
# Clean the heart dataset (4 code cells):
#
# 2d. Fix fbs and exang columns — they contain BOTH string 'True'/'False'
#     AND boolean True/False. Map all values to integer 1/0.
#     Print before/after dtype and value counts for fbs.
#
# 2e. Replace cholesterol (chol) zeros with NaN (biologically impossible).
#     Print zero count before and missing count after.
#
# 2f. Impute missing values:
#     - Numeric columns [trestbps, chol, thalch, oldpeak, ca] → median
#     - Categorical columns [fbs, exang, restecg, slope, thal] → mode
#     Print which columns were filled and with what value.
#
# 2g. Create binary target: `has_heart_disease` = (num > 0).astype(int).
#     Encode sex: `sex_encoded` = 1 if sex == 'Male', else 0.
#     LabelEncode columns [cp, restecg, slope, thal] → new col named
#     col + '_encoded' (e.g., cp_encoded). Store encoders in a dict.
#     Drop columns 'id' and 'dataset'.
#     Print final shape and remaining missing count.

In [48]:
# Fix fbs and exang columns (mixed string/boolean values)
print("="*50)
print("FIXING FBS AND EXANG COLUMNS")
print("="*50)

# Show fbs BEFORE
print("\nFBS - Before:")
print(f"Data type: {heart['fbs'].dtype}")
print("Value counts:")
print(heart['fbs'].value_counts(dropna=False))

# Map fbs: handle both string and boolean values, then fill NaN with 0
heart['fbs'] = heart['fbs'].map({True: 1, False: 0, 'True': 1, 'False': 0}).fillna(0).astype(int)

# Map exang: handle both string and boolean values, then fill NaN with 0
heart['exang'] = heart['exang'].map({True: 1, False: 0, 'True': 1, 'False': 0}).fillna(0).astype(int)

# Show fbs AFTER
print("\nFBS - After:")
print(f"Data type: {heart['fbs'].dtype}")
print("Value counts:")
print(heart['fbs'].value_counts())

# Show exang info
print("\nEXANG - After:")
print(f"Data type: {heart['exang'].dtype}")
print("Value counts:")
print(heart['exang'].value_counts())

FIXING FBS AND EXANG COLUMNS

FBS - Before:
Data type: object
Value counts:
fbs
False    692
True     138
NaN       90
Name: count, dtype: int64

FBS - After:
Data type: int64
Value counts:
fbs
0    782
1    138
Name: count, dtype: int64

EXANG - After:
Data type: int64
Value counts:
exang
0    583
1    337
Name: count, dtype: int64


### ✏️ Cleaning Summary

| Dataset | Action | Details |
|---------|--------|--------|
| Anesthesia | Parsed SurgeryDuration | String → integer |
| Anesthesia | Handled missing Complications | Created binary flag (73 NaN) |
| Anesthesia | Encoded categoricals | Gender, AnesthesiaType, SurgeryType |
| Heart | Fixed fbs/exang types | String True/False → binary int |
| Heart | Fixed cholesterol zeros | 0 → NaN → median imputed |
| Heart | Imputed missing values | Median for numeric, mode for categorical |
| Heart | Created binary target | num > 0 → has_heart_disease |

---
## Step 3 — Enrich the Clinical Notes

The raw notes have only 2 unique values each. That's too limited for meaningful NLP.

We'll build generators that produce **realistic, varied clinical notes** grounded in each patient's actual data. This teaches data augmentation and creates a proper NLP challenge.

**Key principle:** Because we generate the notes, we KNOW the ground truth — which gives us a way to validate our extraction later.

In [ ]:
def generate_preop_note(row):
    """Generate a realistic preoperative note based on patient's structured data."""
    random.seed(row.name + 42)  # Reproducible per patient
    parts = []
    
    # --- Comorbidities based on original note ---
    if 'Hypertension' in row['PreoperativeNotes']:
        parts.append(random.choice([
            'History of hypertension',
            'HTN, managed with medication',
            'Elevated BP, on lisinopril 10mg',
            'Known hypertensive patient, BP controlled',
            'Chronic hypertension, on amlodipine',
            'HTN diagnosed 5 years ago'
        ]))
        parts.append(random.choice([
            'Type 2 diabetes mellitus',
            'DM controlled with metformin',
            'Diabetic, HbA1c 7.2',
            'History of diabetes, on oral hypoglycemics',
            'T2DM, diet controlled',
            'Diabetes mellitus, insulin dependent'
        ]))
    else:
        parts.append(random.choice([
            'No significant past medical history',
            'Generally healthy',
            'No chronic conditions noted',
            'Unremarkable medical history',
            'No prior hospitalizations',
            'Healthy, active lifestyle'
        ]))
    
    # --- BMI-based observations ---
    if row['BMI'] >= 30:
        parts.append(random.choice([
            'Obesity noted, BMI elevated',
            f'Obese, BMI {row["BMI"]}',
            'Weight management counseling provided',
            'Elevated BMI, increased surgical risk discussed'
        ]))
    elif row['BMI'] >= 25:
        if random.random() < 0.3:  # Only sometimes mentioned
            parts.append('Mildly overweight')
    
    # --- Age-based observations ---
    if row['Age'] >= 65:
        parts.append(random.choice([
            'Elderly patient, fall risk assessed',
            'Age over 65, cardiac clearance obtained',
            'Geriatric assessment complete',
            'Advanced age, additional monitoring planned'
        ]))
    elif row['Age'] >= 55:
        if random.random() < 0.3:
            parts.append('Middle-aged, routine screening current')
    
    # --- Surgery-specific ---
    if row['SurgeryType'] == 'Cardiovascular':
        parts.append(random.choice([
            'Cardiac history reviewed',
            'ECG shows normal sinus rhythm',
            'Pre-op stress test completed',
            'Echo shows preserved EF',
            'Cardiology clearance obtained'
        ]))
    elif row['SurgeryType'] == 'Orthopedic':
        if random.random() < 0.4:
            parts.append(random.choice([
                'Orthopedic pre-op checklist complete',
                'Musculoskeletal assessment done'
            ]))
    
    # --- Allergies ---
    parts.append(random.choice([
        'Allergies: NKDA',
        'No known drug allergies',
        'Allergies: Penicillin',
        'Allergies: sulfa drugs',
        'Allergies: latex',
        'NKDA per patient report'
    ]))
    
    # --- Lab results ---
    parts.append(random.choice([
        'Labs WNL',
        'CBC and BMP within normal limits',
        'Pre-op labs reviewed, no concerns',
        'Mild anemia noted on labs',
        'Labs unremarkable',
        'Hemoglobin 13.5, platelets adequate'
    ]))
    
    return '. '.join(parts) + '.'


# Apply to all patients
anes['enriched_preop'] = anes.apply(generate_preop_note, axis=1)

print(f'Unique enriched preop notes: {anes["enriched_preop"].nunique()}')
print(f'(Original had only {anes["PreoperativeNotes"].nunique()} unique values)')
print()
print('--- Sample enriched notes ---')
for i in [0, 1, 50, 100, 200]:
    print(f'\nPatient {i} (original: "{anes.iloc[i]["PreoperativeNotes"]}"):')
    print(f'  → {anes.iloc[i]["enriched_preop"]}')

In [ ]:
def generate_postop_note(row):
    """Generate a realistic postoperative note based on patient's structured data."""
    random.seed(row.name + 99)  # Different seed from preop
    parts = []
    
    # --- Pain based on PostoperativeNotes + PainLevel ---
    if 'Pain, slow recovery' in row['PostoperativeNotes']:
        parts.append(random.choice([
            f'Patient reports pain level {row["PainLevel"]}/10',
            'Significant postoperative pain',
            'Pain requiring IV analgesics',
            'Moderate to severe pain at incision site',
            f'Pain score {row["PainLevel"]}, PCA initiated'
        ]))
        parts.append(random.choice([
            'Recovery slower than expected',
            'Delayed ambulation',
            'Extended PACU stay',
            'Patient requires additional monitoring',
            'Slow to return to baseline'
        ]))
    else:
        parts.append(random.choice([
            'Minimal postoperative pain',
            f'Pain well-controlled, level {row["PainLevel"]}/10',
            'Comfortable post-procedure',
            'Tolerating pain with oral medication',
            'Pain managed effectively'
        ]))
        parts.append(random.choice([
            'No complications observed',
            'Uncomplicated recovery',
            'Recovering as expected',
            'Stable in recovery room'
        ]))
    
    # --- Complications ---
    if pd.notna(row['Complications']):
        comp = row['Complications']
        if 'Respiratory' in comp:
            parts.append(random.choice([
                'Respiratory distress noted, O2 supplementation started',
                'Mild dyspnea post-extubation',
                'Respiratory complication, monitoring SpO2'
            ]))
        elif 'Nausea' in comp:
            parts.append(random.choice([
                'Nausea and vomiting, ondansetron given',
                'PONV managed with antiemetics',
                'Mild bleeding at surgical site, nausea reported'
            ]))
        elif 'Delayed' in comp:
            parts.append(random.choice([
                'Delayed recovery, extended observation',
                'Prolonged emergence from anesthesia',
                'Slower than expected return of reflexes'
            ]))
    
    # --- Vitals ---
    parts.append(random.choice([
        'Vitals stable',
        'Hemodynamically stable post-op',
        'BP and HR within normal limits',
        'Stable vital signs throughout recovery'
    ]))
    
    return '. '.join(parts) + '.'


anes['enriched_postop'] = anes.apply(generate_postop_note, axis=1)

print(f'Unique enriched postop notes: {anes["enriched_postop"].nunique()}')
print(f'(Original had only {anes["PostoperativeNotes"].nunique()} unique values)')
print()
print('--- Sample enriched postop notes ---')
for i in [0, 1, 50, 100, 200]:
    print(f'\nPatient {i} (original: "{anes.iloc[i]["PostoperativeNotes"]}", complications: {anes.iloc[i]["Complications"]}):')
    print(f'  → {anes.iloc[i]["enriched_postop"]}')

---
## Step 4 — Build the NLP Extraction Pipeline

Now we build a system that reads clinical notes and extracts structured features.

**Three components:** text preprocessing → regex-based entity extraction → feature columns.

In [ ]:
# Prompt for Claude Code:
#
# Build the NLP extraction pipeline (5 code cells):
#
# 4a. Text preprocessing function `preprocess_note(note)`:
#     - Lowercase the text
#     - Remove all characters except a-z, 0-9, whitespace, and .,;/
#     - Normalize multiple whitespace to single space, strip
#     Apply to anes['enriched_preop'] → new column 'preop_clean'
#     Apply to anes['enriched_postop'] → new column 'postop_clean'
#     Print a before/after example.
#
# 4b. Define a dict `comorbidity_patterns` with regex patterns using \b word
#     boundaries and clinical synonyms. Must match the vocabulary from Step 3:
#     - hypertension: hypertension|htn|elevated bp|hypertensive|high blood pressure|amlodipine|lisinopril
#     - diabetes: diabetes|dm|diabetic|type 2|metformin|hba1c|hypoglycemic|insulin dependent|t2dm
#     - obesity: obese/obesi|bmi elevated|elevated bmi|high bmi|overweight|weight management
#     - cardiac: cardiac|ecg|echo|stress test|cardiology|sinus rhythm|preserved ef|lvh
#     - anemia: anemia|anemic|low hemoglobin|low hgb
#     - elderly_risk: elderly|geriatric|fall risk|age over 65|advanced age
#     Also define `allergy_positive_pattern` matching 'allergies: penicillin/sulfa/latex'
#     and `allergy_none_pattern` matching 'nkda' or 'no known drug allerg'.
#
# 4c. Function `extract_comorbidities(note)` that:
#     - Searches each comorbidity pattern → binary flag named `nlp_{condition}`
#     - Checks allergy: flag `nlp_has_allergy` = 1 only for specific allergies (NOT NKDA)
#     - Returns a dict of binary flags
#     Apply to anes['preop_clean'], expand to columns, concat onto anes.
#     Print extracted feature counts sorted descending.
#
# 4d. Define `postop_patterns` dict and `extract_postop(note)` function:
#     - nlp_pain: pain|analgesic|pca|hurts|discomfort|pain level \d
#     - nlp_nausea: nausea|vomiting|ponv|antiemetic|ondansetron
#     - nlp_respiratory: respiratory|dyspnea|o2|spo2|extubation|breathing
#     - nlp_slow_recovery: slow|delayed|prolonged|extended|longer than expected
#     - nlp_stable: stable|uncomplicated|well.controlled|no complications|comfortable
#     Apply to postop_clean, expand to columns, concat onto anes.
#
# 4e. Spot-check patients [0, 1, 42, 150, 250]:
#     For each, print their preop note snippet (first 80 chars) and all
#     nlp_ columns where value == 1.

---
## Step 5 — Validate Your NLP

Because we generated the notes, we know the ground truth. Let's measure extraction accuracy.

In [ ]:
# Prompt for Claude Code:
#
# Validate NLP extraction accuracy (3 code cells):
#
# 5a. Build gold-standard labels from the generation logic in Step 3.
#     Create a DataFrame `gold` with these binary columns:
#     - hypertension: 1 if PreoperativeNotes contains 'Hypertension'
#     - diabetes: 1 if PreoperativeNotes contains 'diabetes'
#     - obesity: 1 if BMI >= 30
#     - cardiac: 1 if SurgeryType == 'Cardiovascular'
#     - elderly_risk: 1 if Age >= 65
#     - pain: 1 if PostoperativeNotes contains 'Pain, slow'
#     - slow_recovery: 1 if PostoperativeNotes contains 'slow recovery'
#     - has_complication: copy of anes['has_complications']
#     Print the sum of each gold label.
#
# 5b. For each of these 5 entity pairs, print a full classification_report
#     comparing gold labels to NLP-extracted labels:
#       (hypertension, nlp_hypertension), (diabetes, nlp_diabetes),
#       (obesity, nlp_obesity), (cardiac, nlp_cardiac),
#       (elderly_risk, nlp_elderly_risk)
#     Collect precision/recall/F1/support for the positive class (label '1')
#     into a summary DataFrame called `accuracy_df`.
#
# 5c. Error analysis: for each entity pair above, compute false positive
#     count (NLP=1, gold=0) and false negative count (NLP=0, gold=1).
#     For entities with errors, print 2 sample note snippets (first 100 chars)
#     for each error type (FP and FN).

### ✏️ NLP Validation Notes

**Which entities had the best extraction accuracy? Why?**

*Your answer:*

**Which entities had errors? What caused the false positives/negatives?**

*Your answer:*

**What would you fix if you had another iteration?**

*Your answer:*

---
## Step 6 — Exploratory Data Analysis

Now that we have both structured and NLP-extracted features, let's explore patterns.

In [ ]:
# Prompt for Claude Code:
#
# Exploratory Data Analysis — generate 5 visualization cells:
#
# 6a. 2x2 subplot grid of anesthesia distributions:
#     Top-left: Age histogram. Top-right: BMI histogram.
#     Bottom-left: duration_hours histogram. Bottom-right: PainLevel histogram.
#
# 6b. Side-by-side bar charts (1x2 subplots):
#     Left: Outcome proportion by SurgeryType (use pd.crosstab normalized by index).
#     Right: Outcome proportion by nlp_hypertension (crosstab, normalized).
#
# 6c. Horizontal bar chart showing NLP feature prevalence (mean of all nlp_ columns).
#     Sort ascending. Color-code: blue for preop features, orange for postop
#     features (nlp_pain, nlp_nausea, nlp_respiratory, nlp_slow_recovery, nlp_stable).
#
# 6d. Overlapping histograms comparing anes['Age'] and heart['age'] distributions
#     on the same axes with alpha transparency. Print min-max and mean for both
#     datasets, plus the age overlap zone.
#
# 6e. Seaborn correlation heatmap for heart dataset columns:
#     [age, trestbps, chol, thalch, oldpeak, ca, has_heart_disease].
#     Use annot=True, cmap='RdBu_r', center=0, fmt='.2f', square=True.

### ✏️ EDA Observations

Write at least 5 observations:

1. 
2. 
3. 
4. 
5. 

---
## Step 7 — Merge Datasets & Feature Fusion

We'll merge on age bins and define three feature sets for comparison.

In [ ]:
# Prompt for Claude Code:
#
# Merge datasets and define feature sets (3 code cells):
#
# 7a. Create 5-year age bins using pd.cut with bins=range(25, 80, 5).
#     Add 'age_bin' column to both anes (from 'Age') and heart (from 'age').
#     Print the bin distribution for each dataset sorted by index.
#
# 7b. Inner merge anes and heart on 'age_bin' with suffixes=('_surg', '_heart').
#     Print row counts for anes, heart, and merged. Print merge expansion factor
#     (merged rows / anes rows) and total column count.
#
# 7c. Define exactly these three feature lists:
#
#     structured_features = [
#         'Age', 'BMI', 'duration_hours', 'PainLevel', 'gender_encoded',
#         'anesthesia_general', 'has_complications',
#         'surgery_Cardiovascular', 'surgery_Cosmetic',
#         'surgery_Neurological', 'surgery_Orthopedic',
#     ]
#
#     nlp_features = [
#         'nlp_hypertension', 'nlp_diabetes', 'nlp_obesity', 'nlp_cardiac',
#         'nlp_anemia', 'nlp_elderly_risk', 'nlp_has_allergy',
#         'nlp_pain', 'nlp_nausea', 'nlp_respiratory', 'nlp_slow_recovery',
#         'nlp_stable',
#     ]
#
#     fused_features = structured_features + nlp_features
#
#     Print the count for each set. Assert every column exists in anes.

---
## Step 8 — Statistical Testing

Before modeling, let's test whether NLP-extracted features are associated with outcomes.

In [ ]:
# Prompt for Claude Code:
#
# Statistical hypothesis testing (1 code cell):
#
# Chi-square tests (chi2_contingency on pd.crosstab) for these pairs:
#   - nlp_hypertension vs Outcome
#   - nlp_diabetes vs Outcome
#   - nlp_obesity vs Outcome
#   - nlp_cardiac vs Outcome
#
# Independent t-tests (ttest_ind) for:
#   - PainLevel grouped by nlp_hypertension (group 1 vs group 0)
#   - PainLevel grouped by nlp_diabetes (group 1 vs group 0)
#
# For each test, print: test name, statistic, p-value, and whether
# significant at p < 0.05. Also print group means for t-tests.
# Collect all results into a summary DataFrame called `results_df`.

---
## Step 9 — The Three-Way Model Comparison

This is the payoff. We train models on structured-only, NLP-only, and fused feature sets, then compare. **The key metric: does adding NLP features improve prediction?**

In [ ]:
# Prompt for Claude Code:
#
# Three-way model comparison (4 code cells):
#
# 9a. Set target = 'Outcome'. Create y from anes[target].
#     Build a dict `feature_sets` with keys 'Structured', 'NLP-only', 'Fused'
#     mapping to structured_features, nlp_features, fused_features.
#     Split anes[fused_features] into train/test with test_size=0.2,
#     stratify=y, random_state=42. Print sizes and class balance.
#
# 9b. Loop over each feature set and train both:
#     - LogisticRegression(max_iter=1000, random_state=42)
#     - RandomForestClassifier(n_estimators=100, random_state=42)
#     For each, compute: accuracy, weighted precision/recall/F1, AUC.
#     Store ROC curve data (fpr, tpr, auc) in a dict `roc_data` keyed
#     by '{feature_set} — {model_name}'.
#     Save the Fused RandomForest model as `fused_rf`.
#     Collect all metrics into `comparison_df` DataFrame and display it.
#
# 9c. Plot all 6 ROC curves on one figure (10x8).
#     Color by feature set: Structured=#2E75B6, NLP-only=#C55A11, Fused=#548235.
#     Line style by model: LogReg=solid, RandomForest=dashed.
#     Include diagonal random baseline. Legend in lower right.
#
# 9d. AUC bar chart: pivot comparison_df by Feature Set × Model for AUC.
#     Order: Structured, NLP-only, Fused. Add value labels on bars.
#     Draw horizontal line at 0.5 for random baseline.
#     Calculate NLP uplift = best Fused AUC − best Structured AUC.
#     Print the uplift prominently.

---
## Step 10 — Feature Importance & Clinical Interpretation

In [ ]:
# Prompt for Claude Code:
#
# Feature importance visualization (1 code cell):
#
# Extract feature_importances_ from fused_rf (the Fused RandomForest
# model saved in Step 9b).
# Create a DataFrame with columns: feature, importance, source
# (source = 'NLP' if feature starts with 'nlp_', else 'Structured').
# Sort by importance ascending.
#
# Plot a horizontal bar chart (10x8) of all fused features.
# Color-code: blue (#2E75B6) for Structured, orange (#C55A11) for NLP.
# Add a legend using matplotlib.patches.Patch.
# Print how many NLP features appear in the top 10 most important.

### ✏️ Interpretation

**Does NLP add value?** Quantify the uplift and explain what it means.

*Your answer:*

**Which NLP features contributed most?** Why do these make clinical sense?

*Your answer:*

**Synthetic data limitation:** Be honest about what this means for real-world applicability.

*Your answer:*

---
## Step 11 — NLP Pipeline Documentation & Error Analysis

### Pipeline Diagram

```
Raw Clinical Notes (2 unique values)
        │
        ▼
Note Enrichment (generate_preop_note / generate_postop_note)
        │  Uses: Age, BMI, SurgeryType, original note, Complications
        ▼
Enriched Notes (50+ unique variants)
        │
        ▼
Text Preprocessing (lowercase, clean punctuation, normalize whitespace)
        │
        ▼
Regex Entity Extraction (comorbidity_patterns + postop_patterns)
        │  Handles: synonyms, word boundaries, allergy negation
        ▼
NLP Feature Columns (binary flags: nlp_hypertension, nlp_diabetes, ...)
        │
        ▼
Validation (gold standard → precision / recall / F1 → error analysis → iterate)
        │
        ▼
Feature Fusion (structured_features + nlp_features → fused_features)
        │
        ▼
Three-Way Model Comparison → NLP Uplift Quantified
```

### Scalability Discussion

**Would this regex pipeline work on 10,000 real clinical notes?**

*Your answer:*

**What would a production NLP system need beyond regex?**

Consider: negation detection (NegEx), clinical NER models (MedSpaCy, Clinical BERT), abbreviation expansion, section detection (HPI vs. Assessment vs. Plan).

*Your answer:*

### Rule-Based vs. ML-Based NLP

| Aspect | Rule-Based (our regex) | ML-Based (spaCy / BERT) |
|--------|------------------------|-------------------------|
| Setup effort | | |
| Accuracy on known patterns | | |
| Handling new/unseen text | | |
| Negation handling | | |
| Maintenance | | |
| Speed | | |

*(Fill in the table with your assessment)*

---
## Step 12 — Conclusion & Limitations

### Key Findings

*(Fill in after completing the project)*

1. 
2. 
3. 
4. 
5. 

### Limitations

- The enriched notes were generated FROM structured data, so NLP features are partially derivative. In a real system with genuine clinical notes, NLP would capture information (medication dosages, family history, social history) that doesn't exist in any structured field.
- The dataset merge is approximate (age-binned, not patient-matched).
- The anesthesia dataset is small (300 patients).
- *(Add your own additional limitations)*

### Future Work

- 
- 
- 

---

*Go back to the top and fill in the Executive Summary now that you know your results.*